In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

In [2]:
# 1. CONFIGURACIÓN Y RUTAS
directorio = '/datos_raw'

archivos = {
    'epidemiology': f'{directorio}/epidemiology.csv',
    'vaccinations': f'{directorio}/vaccinations.csv',
    'hospitalizations': f'{directorio}/hospitalizations.csv',
    'demographics': f'{directorio}/demographics.csv',
    'geography': f'{directorio}/geography.csv',
}

# Países a procesar: Chile, Brasil, Argentina, Perú
archivos = ['CL', 'BR', 'AR', 'PE']
paises = {'CL': 'Chile', 'BR': 'Brasil', 'AR': 'Argentina', 'PE': 'Perú'}

OUTPUT_FILE = f'{directorio}/dashboard_data_multipaís.csv'

In [3]:
  # 2. FUNCIONES AUXILIARES

def normalize_date(df):
    """Convierte columna 'date' a datetime si no lo es."""
    if df['date'].dtype != 'datetime64[ns]':
        df['date'] = pd.to_datetime(df['date'])
    return df


def calculate_rate_per_100k(value, population):
    """Calcula tasa por 100.000 habitantes, maneja NaN."""
    return np.where(
        (value.notna()) & (population.notna()) & (population > 0),
        (value / population) * 100000,
        np.nan
    )


def rolling_7day_mean(series):
    """Calcula media móvil de 7 días."""
    return series.rolling(window=7, min_periods=1, center=False).mean()


def apply_lag_days(series, lag_days):
    """Aplica desfase temporal en días. Negativo = rezago (backward shift)."""
    return series.shift(periods=-lag_days)


def normalize_0_to_100(series):
    """Normaliza serie a rango [0, 100] usando min-max scaling."""
    min_val = series.min()
    max_val = series.max()
    if max_val == min_val:
        return pd.Series([50] * len(series), index=series.index)
    return (series - min_val) / (max_val - min_val) * 100


def country_code_to_name(location_key):
    """Extrae código de país de location_key."""
    if '_' in str(location_key):
        return str(location_key).split('_')[0]
    return str(location_key)


def region_code_to_name(location_key):
    """Mapea códigos de región (XX_YY) a nombres descriptivos por país."""
    # Mapeo para Chile
    cl_map = {
        'CL': 'Chile (Nacional)',
        'CL_AI': 'Arica y Parinacota',
        'CL_AN': 'Antofagasta',
        'CL_AP': 'Atacama',
        'CL_AR': 'Araucanía',
        'CL_AT': 'Antofagasta Alt.',
        'CL_BI': 'Bío-Bío',
        'CL_CO': 'Los Ríos',
        'CL_LI': 'Libertador General Bernardo O\'Higgins',
        'CL_LL': 'Los Lagos',
        'CL_LR': 'Los Lagos Rural',
        'CL_MA': 'Maule',
        'CL_ML': 'Metropolitana',
        'CL_NB': 'Ñuble',
        'CL_RM': 'Región Metropolitana',
        'CL_TA': 'Tarapacá',
        'CL_VS': 'Valparaíso',
    }

    # Mapeo para Brasil (estados principales)
    br_map = {
        'BR': 'Brasil (Nacional)',
    }

    # Mapeo para Argentina (provincias principales)
    ar_map = {
        'AR': 'Argentina (Nacional)',
    }

    # Mapeo para Perú (regiones principales)
    pe_map = {
        'PE': 'Perú (Nacional)',
    }

    # Combinar todos los mapeos
    all_maps = {**cl_map, **br_map, **ar_map, **pe_map}
    return all_maps.get(location_key, location_key)

In [4]:
print(" Leyendo archivos CSV...")

# Epidemiology: casos, fallecidos, recuperados
df_epi = pd.read_csv(archivos['epidemiology'])
df_epi = normalize_date(df_epi)

# Vaccinations: dosis administradas, cobertura por marca
df_vac = pd.read_csv(archivos['vaccinations'])
df_vac = normalize_date(df_vac)

# Hospitalizations: UCI, ventiladores, camas
df_hosp = pd.read_csv(archivos['hospitalizations'])
df_hosp = normalize_date(df_hosp)

# Demographics: población por región
df_demo = pd.read_csv(archivos['demographics'])
# Geography: coordenadas y nombre de regiones
df_geo = pd.read_csv(archivos['geography'])

print(f"  ✓ Epidemiology: {df_epi.shape[0]} filas")
print(f"  ✓ Vaccinations: {df_vac.shape[0]} filas")
print(f"  ✓ Hospitalizations: {df_hosp.shape[0]} filas")
print(f"  ✓ Demographics: {df_demo.shape[0]} filas")
print(f"  ✓ Geography: {df_geo.shape[0]} filas")

 Leyendo archivos CSV...


TypeError: list indices must be integers or slices, not str

In [ ]:
print("\n Filtrando por países seleccionados (Chile, Brasil, Argentina, Perú)...")

# Crear máscara para filtrar múltiples países
country_mask = df_epi['location_key'].str[:2].isin(paises)

# Filtrar por los 4 países (nacional + regiones)
df_epi = df_epi[country_mask].copy()
df_vac = df_vac[df_vac['location_key'].str[:2].isin(paises)].copy()
df_hosp = df_hosp[df_hosp['location_key'].str[:2].isin(paises)].copy()
df_demo = df_demo[df_demo['location_key'].str[:2].isin(paises)].copy()
df_geo = df_geo[df_geo['location_key'].str[:2].isin(paises)].copy()

print(f"   Epidemiology: {df_epi.shape[0]} filas después del filtro")
print(f"   Vaccinations: {df_vac.shape[0]} filas después del filtro")
print(f"   Hospitalizations: {df_hosp.shape[0]} filas después del filtro")

In [ ]:
print("\n Preparando dimensiones demográficas...")

# Seleccionar columnas relevantes de demographics
df_demo = df_demo[['location_key', 'population']].copy()
df_demo = df_demo[df_demo['population'].notna()]

# Seleccionar columnas de geografía
df_geo = df_geo[['location_key', 'latitude', 'longitude']].copy()

print(f"  ✓ Demographics usable: {df_demo.shape[0]} regiones con población")
print(f"  ✓ Geography usable: {df_geo.shape[0]} regiones con coordenadas")

#Preparación Epidemiológica

In [ ]:
print("\n Preparando indicadores epidemiológicos...")

# Seleccionar columnas clave
df_epi = df_epi[[
    'date',
    'location_key',
    'new_confirmed',
    'new_deceased',
    'cumulative_confirmed',
    'cumulative_deceased'
]].copy()

# Merge con población para calcular tasas
df_epi = df_epi.merge(df_demo, on='location_key', how='left')

# Calcular tasas diarias por 100.000 habitantes
df_epi['cases_per_100k_daily'] = calculate_rate_per_100k(
    df_epi['new_confirmed'],
    df_epi['population']
)

df_epi['deaths_per_100k_daily'] = calculate_rate_per_100k(
    df_epi['new_deceased'],
    df_epi['population']
)

# Calcular medias móviles de 7 días (indicador de tendencia)
df_epi = df_epi.sort_values(['location_key', 'date'])
df_epi['cases_per_100k_7d'] = df_epi.groupby('location_key')[
    'cases_per_100k_daily'
].transform(rolling_7day_mean)

df_epi['deaths_per_100k_7d'] = df_epi.groupby('location_key')[
    'deaths_per_100k_daily'
].transform(rolling_7day_mean)

# Indicador de severidad (proxy: ratio muertes / casos confirmados, en %)
df_epi['case_fatality_rate_pct'] = np.where(
    (df_epi['new_confirmed'] > 0),
    (df_epi['new_deceased'] / df_epi['new_confirmed'] * 100).fillna(0),
    np.nan
)

print("   Tasas diarias calculadas")
print("   Medias móviles 7d calculadas")
print("   Severidad (CFR) calculada")

#Preparación Vacunaciones

In [ ]:
print("\n💉 Preparando indicadores de vacunación...")

# Seleccionar columnas clave (dosis y cobertura)
df_vac = df_vac[[
    'date',
    'location_key',
    'cumulative_vaccine_doses_administered',
    'cumulative_persons_fully_vaccinated',
    'cumulative_vaccine_doses_administered_pfizer',
    'cumulative_vaccine_doses_administered_moderna',
    'cumulative_vaccine_doses_administered_janssen',
    'total_vaccine_doses_administered_sinovac',  # nota: algunos son "total_" no "cumulative_"
]].copy()

# Merge con población
df_vac = df_vac.merge(df_demo, on='location_key', how='left')

# Calcular tasa de dosis por 100.000
df_vac['vaccine_doses_per_100k'] = calculate_rate_per_100k(
    df_vac['cumulative_vaccine_doses_administered'],
    df_vac['population']
)

# Calcular cobertura de vacunación completa (porcentaje)
df_vac['vaccination_coverage_pct'] = np.where(
    (df_vac['cumulative_persons_fully_vaccinated'].notna()) & (df_vac['population'] > 0),
    (df_vac['cumulative_persons_fully_vaccinated'] / df_vac['population']) * 100,
    np.nan
)

# Limitar a máximo 100%
df_vac['vaccination_coverage_pct'] = df_vac['vaccination_coverage_pct'].clip(upper=100)

# Calcular dosis diarias derivando de acumulado
df_vac = df_vac.sort_values(['location_key', 'date'])
df_vac['vaccine_doses_daily'] = df_vac.groupby('location_key')[
    'cumulative_vaccine_doses_administered'
].diff().fillna(0)

df_vac['vaccine_doses_per_100k_daily'] = calculate_rate_per_100k(
    df_vac['vaccine_doses_daily'],
    df_vac['population']
)

# Media móvil 7d de dosis diarias
df_vac['vaccine_doses_per_100k_7d'] = df_vac.groupby('location_key')[
    'vaccine_doses_per_100k_daily'
].transform(rolling_7day_mean)

# Descomponer por laboratorio (tasa diaria de dosis nuevas)
for brand, col in [
    ('Pfizer', 'cumulative_vaccine_doses_administered_pfizer'),
    ('Moderna', 'cumulative_vaccine_doses_administered_moderna'),
    ('Janssen', 'cumulative_vaccine_doses_administered_janssen'),
    ('Sinovac', 'total_vaccine_doses_administered_sinovac'),
]:
    df_vac[f'doses_daily_{brand.lower()}'] = df_vac.groupby('location_key')[
        col
    ].diff().fillna(0)
    df_vac[f'doses_per_100k_daily_{brand.lower()}'] = calculate_rate_per_100k(
        df_vac[f'doses_daily_{brand.lower()}'],
        df_vac['population']
    )

print("   Tasa de dosis calculada")
print("   Cobertura de vacunación calculada")
print("   Descomposición por laboratorio completada")

#Preparación Hospitalizaciones

In [ ]:
rint("\n Preparando indicadores de hospitalización...")

# Seleccionar columnas clave
df_hosp = df_hosp[[
    'date',
    'location_key',
    'current_hospitalized_patients',
    'current_intensive_care_patients',
    'current_ventilator_patients',
    'new_hospitalized_patients',
    'new_intensive_care_patients',
]].copy()

# Merge con población
df_hosp = df_hosp.merge(df_demo, on='location_key', how='left')

# Calcular tasas por 100.000
df_hosp['hospitalized_per_100k'] = calculate_rate_per_100k(
    df_hosp['current_hospitalized_patients'],
    df_hosp['population']
)

df_hosp['icu_per_100k'] = calculate_rate_per_100k(
    df_hosp['current_intensive_care_patients'],
    df_hosp['population']
)

df_hosp['ventilator_per_100k'] = calculate_rate_per_100k(
    df_hosp['current_ventilator_patients'],
    df_hosp['population']
)

df_hosp['new_hospitalized_per_100k_daily'] = calculate_rate_per_100k(
    df_hosp['new_hospitalized_patients'],
    df_hosp['population']
)

df_hosp['new_icu_per_100k_daily'] = calculate_rate_per_100k(
    df_hosp['new_intensive_care_patients'],
    df_hosp['population']
)

# Media móvil 7d de nuevos ingresos
df_hosp = df_hosp.sort_values(['location_key', 'date'])
df_hosp['new_hospitalized_per_100k_7d'] = df_hosp.groupby('location_key')[
    'new_hospitalized_per_100k_daily'
].transform(rolling_7day_mean)

df_hosp['new_icu_per_100k_7d'] = df_hosp.groupby('location_key')[
    'new_icu_per_100k_daily'
].transform(rolling_7day_mean)

print("  Tasas de hospitalización calculadas")
print("  Tasas de UCI calculadas")
print("  Medias móviles calculadas")

#Unión de todos los indicadores

In [ ]:
print("\n Fusionando indicadores epidemiológicos + vacunaciones + hospitalizaciones...")

# Empezar con epidemiología
df_master = df_epi[[
    'date',
    'location_key',
    'population',
    'cases_per_100k_daily',
    'deaths_per_100k_daily',
    'cases_per_100k_7d',
    'deaths_per_100k_7d',
    'case_fatality_rate_pct',
]].copy()

# Añadir vacunaciones
df_vac_cols = df_vac[[
    'date',
    'location_key',
    'vaccine_doses_per_100k',
    'vaccination_coverage_pct',
    'vaccine_doses_per_100k_7d',
    'doses_per_100k_daily_pfizer',
    'doses_per_100k_daily_moderna',
    'doses_per_100k_daily_janssen',
    'doses_per_100k_daily_sinovac',
]]

df_master = df_master.merge(
    df_vac_cols,
    on=['date', 'location_key'],
    how='outer'
)

# Rellenar población si falta
df_master = df_master.merge(df_demo, on='location_key', how='left')
df_master['population'] = df_master['population_x'].fillna(df_master['population_y'])
df_master = df_master.drop(['population_x', 'population_y'], axis=1)

# Añadir hospitalizaciones
df_hosp_cols = df_hosp[[
    'date',
    'location_key',
    'hospitalized_per_100k',
    'icu_per_100k',
    'ventilator_per_100k',
    'new_hospitalized_per_100k_7d',
    'new_icu_per_100k_7d',
]]

df_master = df_master.merge(
    df_hosp_cols,
    on=['date', 'location_key'],
    how='left'
)

# Añadir geografía
df_master = df_master.merge(
    df_geo,
    on='location_key',
    how='left'
)

# Añadir nombre legible de región
df_master['region_name'] = df_master['location_key'].apply(region_code_to_name)

print(f"  Dataset maestro: {df_master.shape[0]} filas x {df_master.shape[1]} columnas")

#Creación de variables para gráficos

In [ ]:
print("\n📈 Creando variables derivadas para análisis de impacto...")

# Crear columna de severidad genérica (combinada) usando UCI como proxy principal
df_master['severity_indicator_per_100k'] = df_master['icu_per_100k'].fillna(0) + \
                                            df_master['new_icu_per_100k_7d'].fillna(0) / 2

# Crear variable de desfase temporal: aplicar lag a indicadores de severidad
# Lag de 14 días (efectividad de vacunación se ve 2 semanas después)
LAG_DAYS = 14

df_master = df_master.sort_values(['location_key', 'date'])
df_master['deaths_per_100k_7d_lagged'] = df_master.groupby('location_key')[
    'deaths_per_100k_7d'
].transform(lambda x: apply_lag_days(x, LAG_DAYS))

df_master['icu_per_100k_lagged'] = df_master.groupby('location_key')[
    'icu_per_100k'
].transform(lambda x: apply_lag_days(x, LAG_DAYS))

# Crear variables normalizadas [0-100] para comparar vacunación vs severidad en escala comparable
df_master['vaccination_coverage_normalized'] = df_master.groupby('location_key')[
    'vaccination_coverage_pct'
].transform(normalize_0_to_100)

df_master['deaths_normalized'] = df_master.groupby('location_key')[
    'deaths_per_100k_7d'
].transform(normalize_0_to_100)

df_master['icu_normalized'] = df_master.groupby('location_key')[
    'icu_per_100k'
].transform(normalize_0_to_100)

# Crear período epidemiológico (etiqueta simplificada para análisis por ola)
# Basado en fecha aproximada de ondas COVID-19 en Chile
def assign_wave(date):
    if pd.isna(date):
        return 'Unknown'
    d = pd.Timestamp(date)
    if d < pd.Timestamp('2021-03-01'):
        return 'Pre-Vacunación'
    elif d < pd.Timestamp('2021-08-01'):
        return 'Onda Delta'
    elif d < pd.Timestamp('2022-01-01'):
        return 'Onda Ómicron Inicial'
    elif d < pd.Timestamp('2022-08-01'):
        return 'Post-Ómicron'
    else:
        return 'Endemic'

df_master['epidemic_wave'] = df_master['date'].apply(assign_wave)

print("   Severidad combinada creada")
print("   Variables lagged (14 días) creadas")
print("   Variables normalizadas [0-100] creadas")
print("   Períodos epidemiológicos asignados")

# Limpieza y validación

In [ ]:
print("\n Limpieza final y validación...")

# Ordenar por ubicación y fecha
df_master = df_master.sort_values(['location_key', 'date']).reset_index(drop=True)

# Mostrar resumen de datos faltantes
print("\n  Datos faltantes (%):")
missing_pct = (df_master.isnull().sum() / len(df_master) * 100).sort_values(ascending=False)
print(missing_pct[missing_pct > 0].head(15))

# Resumen de filas por región
print("\n  Filas por región/nivel:")
print(df_master.groupby('region_name')['date'].agg(['count', 'min', 'max']))

#Exportación

In [ ]:
print(f"\n Exportando a {OUTPUT_FILE}...")
df_master.to_csv(OUTPUT_FILE)
print(f"  Dataset guardado exitosamente")

In [ ]:
print("\n ETL completado")
print(f"\n Dataset Final:")
print(f"   - Filas: {df_master.shape[0]:,}")
print(f"   - Columnas: {df_master.shape[1]}")
print(f"   - Rango temporal: {df_master['date'].min().date()} a {df_master['date'].max().date()}")
print(f"   - Regiones: {df_master['location_key'].nunique()}")
print(f"   - Archivo: {OUTPUT_FILE}")

print(f"\n📋 Diccionario de columnas principales:")
cols_desc = {
    'date': 'Fecha del registro',
    'location_key': 'Código de región (CL, CL_RM, etc.)',
    'region_name': 'Nombre descriptivo de región',
    'population': 'Población total de la región',
    'cases_per_100k_7d': '[Gráfico 1] Casos diarios por 100k (media 7d)',
    'deaths_per_100k_7d': '[Gráfico 1] Muertes diarias por 100k (media 7d)',
    'icu_per_100k': '[Gráfico 1] UCI actuales por 100k',
    'vaccine_doses_per_100k_7d': '[Gráfico 2] Dosis diarias por 100k (media 7d)',
    'vaccination_coverage_pct': '[Gráfico 2, 4] Cobertura vacunación completa (%)',
    'vaccination_coverage_normalized': '[Gráfico 2] Cobertura normalizada [0-100]',
    'deaths_normalized': '[Gráfico 2] Muertes normalizadas [0-100]',
    'doses_per_100k_daily_pfizer': '[Gráfico 3] Dosis Pfizer diarias por 100k',
    'doses_per_100k_daily_moderna': '[Gráfico 3] Dosis Moderna diarias por 100k',
    'doses_per_100k_daily_janssen': '[Gráfico 3] Dosis Janssen diarias por 100k',
    'doses_per_100k_daily_sinovac': '[Gráfico 3] Dosis Sinovac diarias por 100k',
    'deaths_per_100k_7d_lagged': '[Gráfico 4] Muertes 14d rezagadas por 100k',
    'icu_per_100k_lagged': '[Gráfico 4] UCI 14d rezagadas por 100k',
    'epidemic_wave': '[Gráfico 4, 5] Período epidemiológico (Pre-Vac, Delta, Ómicron, etc.)',
    'latitude': '[Gráfico 5] Latitud para mapa',
    'longitude': '[Gráfico 5] Longitud para mapa',
}

for col, desc in cols_desc.items():
    print(f"   - {col:<35} : {desc}")
